# EEG Motor İmgelem — Derin Öğrenme Sınıflandırması
## PhysioNet EEGMMIdb | EEGNet vs ShallowConvNet vs DeepConvNet

**Kaynak:** https://www.physionet.org/content/eegmmidb/1.0.0/

### Pipeline
- Ham EEG sinyali → uçtan uca derin öğrenme (özellik çıkarma yok)
- 3 model: **EEGNet**, **ShallowConvNet**, **DeepConvNet**
- Within-subject 5-fold CV (klasik ML ile adil karşılaştırma)
- PCA uygulaması: modelin öğrendiği embedding üzerinde
- Klasik ML sonuçlarıyla nihai karşılaştırma tablosu

## 1. Kurulum ve Kütüphaneler

In [2]:
# Braindecode kütüphanesi EEG derin öğrenme modellerini içerir
# !pip install braindecode mne torch scikit-learn numpy matplotlib seaborn

In [3]:
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

mne.set_log_level('WARNING')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"PyTorch : {torch.__version__}")
print(f"MNE     : {mne.__version__}")
print(f"Cihaz   : {DEVICE}")
print(f"CUDA    : {'Evet' if torch.cuda.is_available() else 'Hayır (CPU ile çalışılacak)'}")

PyTorch : 2.11.0
MNE     : 1.12.1
Cihaz   : cpu
CUDA    : Hayır (CPU ile çalışılacak)


## 2. Parametreler

In [4]:
DATA_DIR     = '../physioNet_Dataset'
N_SUBJECTS   = 40
IMAGERY_RUNS = ['R04', 'R08', 'R12']

# Filtre: Mu + Beta bandı (motor imgelem için kritik)
L_FREQ, H_FREQ = 8.0, 30.0

# Epoch penceresi — saf imgelem dönemi
TMIN, TMAX = 1.0, 4.0

# Derin öğrenme için ham sinyal tutulur (özellik çıkarılmaz)
# Her epoch: (n_channels=64, n_times=160 Hz × 3s = 480)

N_FOLDS    = 5
BATCH_SIZE = 16
N_EPOCHS   = 100     # Eğitim epoch sayısı
LR         = 1e-3    # Learning rate
PATIENCE   = 15      # Early stopping

print(f"Denek       : {N_SUBJECTS}")
print(f"Sinyal      : {L_FREQ}-{H_FREQ} Hz, {TMIN}-{TMAX}s")
print(f"CV          : {N_FOLDS}-fold within-subject")
print(f"Eğitim epok : {N_EPOCHS}  |  Batch: {BATCH_SIZE}  |  LR: {LR}")

Denek       : 40
Sinyal      : 8.0-30.0 Hz, 1.0-4.0s
CV          : 5-fold within-subject
Eğitim epok : 100  |  Batch: 16  |  LR: 0.001


## 3. Veri Yükleme

In [5]:
def load_subject_raw(subject_id, runs, data_dir, l_freq, h_freq, tmin, tmax):
    """
    Ham EEG epoch'larını döndürür: (n_epochs, n_channels, n_times)
    Derin öğrenme için özellik çıkarımı yapılmaz.
    """
    sub  = f'S{subject_id:03d}'
    raws = []
    for run in runs:
        path = f'{data_dir}/{sub}/{sub}{run}.edf'
        try:
            raws.append(mne.io.read_raw_edf(path, preload=True, verbose=False))
        except FileNotFoundError:
            pass
    if not raws:
        return None, None

    raw = mne.concatenate_raws(raws)
    raw.rename_channels({ch: ch.rstrip('.').upper() for ch in raw.ch_names})
    raw.set_montage(mne.channels.make_standard_montage('standard_1020'),
                    on_missing='ignore')
    raw.pick('eeg')
    raw.filter(l_freq, h_freq, fir_design='firwin', verbose=False)
    raw.notch_filter(60, verbose=False)

    events, ann_id = mne.events_from_annotations(raw, verbose=False)
    ev_id = {k: v for k, v in ann_id.items() if k in ['T1', 'T2']}
    if not ev_id:
        return None, None

    ep = mne.Epochs(raw, events, ev_id, tmin=tmin, tmax=tmax,
                    proj=True, picks='eeg', baseline=None,
                    preload=True, verbose=False)
    ep.drop_bad(verbose=False)

    X = ep.get_data().astype(np.float32)   # (n_ep, 64, n_times)
    le = LabelEncoder()
    y  = le.fit_transform(ep.events[:, -1]).astype(np.int64)

    return X, y


print("Yükleme fonksiyonu hazır.")

Yükleme fonksiyonu hazır.


In [6]:
subjects_X = {}   # {sid: ndarray (n_ep, 64, n_times)}
subjects_y = {}   # {sid: ndarray (n_ep,)}

print(f"Toplam {N_SUBJECTS} denek yükleniyor...\n")
for sid in range(1, N_SUBJECTS + 1):
    X, y = load_subject_raw(sid, IMAGERY_RUNS, DATA_DIR,
                            L_FREQ, H_FREQ, TMIN, TMAX)
    if X is None or len(np.unique(y)) < 2 or len(y) < 20:
        continue
    subjects_X[sid] = X
    subjects_y[sid] = y
    print(f"  S{sid:03d}: {X.shape[0]} epoch | {X.shape[1]} kanal | {X.shape[2]} zaman noktası")

n_channels = list(subjects_X.values())[0].shape[1]
n_times    = list(subjects_X.values())[0].shape[2]
print(f"\nYüklenen denek : {len(subjects_X)}")
print(f"Giriş boyutu   : ({n_channels} kanal × {n_times} zaman noktası)")

Toplam 40 denek yükleniyor...

  S001: 45 epoch | 64 kanal | 481 zaman noktası
  S002: 45 epoch | 64 kanal | 481 zaman noktası
  S003: 45 epoch | 64 kanal | 481 zaman noktası
  S004: 45 epoch | 64 kanal | 481 zaman noktası
  S005: 45 epoch | 64 kanal | 481 zaman noktası
  S006: 45 epoch | 64 kanal | 481 zaman noktası
  S007: 45 epoch | 64 kanal | 481 zaman noktası
  S008: 45 epoch | 64 kanal | 481 zaman noktası
  S009: 45 epoch | 64 kanal | 481 zaman noktası
  S010: 45 epoch | 64 kanal | 481 zaman noktası
  S011: 45 epoch | 64 kanal | 481 zaman noktası
  S012: 45 epoch | 64 kanal | 481 zaman noktası
  S013: 45 epoch | 64 kanal | 481 zaman noktası
  S014: 45 epoch | 64 kanal | 481 zaman noktası
  S015: 45 epoch | 64 kanal | 481 zaman noktası
  S016: 45 epoch | 64 kanal | 481 zaman noktası
  S017: 45 epoch | 64 kanal | 481 zaman noktası
  S018: 45 epoch | 64 kanal | 481 zaman noktası
  S019: 45 epoch | 64 kanal | 481 zaman noktası
  S020: 45 epoch | 64 kanal | 481 zaman noktası
  S021: 4

## 4. Derin Öğrenme Modelleri

### EEGNet (Lawhern et al., 2018)
Kompakt ve verimli. Depthwise + Separable Conv ile hem uzamsal hem zamansal filtreleme.
Motor imgelem BCI'da referans model.

### ShallowConvNet (Schirrmeister et al., 2017)
Sığ mimari. Güç spektrum tabanlı CSP özelliklerini CNN ile öğrenir.

### DeepConvNet (Schirrmeister et al., 2017)
Derin mimari. Daha karmaşık uzamsal-temporal hiyerarşi.

> Tüm modeller ham EEG sinyalini alır — özellik çıkarma yok.

In [ ]:
# ─── EEGNet ───────────────────────────────────────────────────────────────
class EEGNet(nn.Module):
    """
    EEGNet: Lawhern et al. (2018)
    Giriş: (batch, 1, n_channels, n_times)
    """
    def __init__(self, n_channels, n_times, n_classes=2,
                 F1=8, D=2, F2=16, dropout=0.5):
        super().__init__()
        # Temporal Conv
        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(F1),
        )
        # Depthwise Conv (uzamsal filtreleme)
        self.block2 = nn.Sequential(
            nn.Conv2d(F1, F1 * D, (n_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout),
        )
        # Separable Conv
        self.block3 = nn.Sequential(
            nn.Conv2d(F1 * D, F2, (1, 16), padding=(0, 8), bias=False),
            nn.Conv2d(F2, F2, 1, bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout),
        )
        # Classifier
        dummy = torch.zeros(1, 1, n_channels, n_times)
        feat_size = self._get_feat_size(dummy)
        self.classifier = nn.Linear(feat_size, n_classes)

    def _get_feat_size(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        return x.numel()

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

    def get_embedding(self, x):
        """PCA için embedding (classifier öncesi özellikler)"""
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        return x.view(x.size(0), -1)


# ─── ShallowConvNet ────────────────────────────────────────────────────────
class ShallowConvNet(nn.Module):
    """
    ShallowConvNet: Schirrmeister et al. (2017)
    CSP benzeri uzamsal filtreler öğrenir.
    """
    def __init__(self, n_channels, n_times, n_classes=2, dropout=0.5):
        super().__init__()
        self.temporal = nn.Conv2d(1, 40, (1, 25), bias=False)
        self.spatial  = nn.Conv2d(40, 40, (n_channels, 1), bias=False)
        self.bn       = nn.BatchNorm2d(40)
        self.pool     = nn.AvgPool2d((1, 75), stride=(1, 15))
        self.drop     = nn.Dropout(dropout)

        dummy     = torch.zeros(1, 1, n_channels, n_times)
        feat_size = self._get_feat_size(dummy)
        self.classifier = nn.Linear(feat_size, n_classes)

    def _get_feat_size(self, x):
        x = self.temporal(x)
        x = self.spatial(x)
        x = self.bn(x)
        x = x ** 2
        x = self.pool(x)
        x = torch.log(torch.clamp(x, min=1e-6))
        return x.numel()

    def forward(self, x):
        x = self.temporal(x)
        x = self.spatial(x)
        x = self.bn(x)
        x = x ** 2
        x = self.pool(x)
        x = torch.log(torch.clamp(x, min=1e-6))
        feat = x.view(x.size(0), -1)
        feat = self.drop(feat)
        return self.classifier(feat)

    def get_embedding(self, x):
        x = self.temporal(x)
        x = self.spatial(x)
        x = self.bn(x)
        x = x ** 2
        x = self.pool(x)
        x = torch.log(torch.clamp(x, min=1e-6))
        return x.view(x.size(0), -1)


# ─── DeepConvNet ──────────────────────────────────────────────────────────
class DeepConvNet(nn.Module):
    """
    DeepConvNet: Schirrmeister et al. (2017)
    Derin hiyerarşik uzamsal-temporal özellik çıkarma.
    """
    def __init__(self, n_channels, n_times, n_classes=2, dropout=0.5):
        super().__init__()
        def conv_block(in_f, out_f, k, pool=True):
            layers = [
                nn.Conv2d(in_f, out_f, (1, k), bias=False),
                nn.BatchNorm2d(out_f),
                nn.ELU(),
                nn.Dropout(dropout),
            ]
            if pool:
                layers.append(nn.MaxPool2d((1, 3), stride=(1, 3)))
            return nn.Sequential(*layers)

        self.block1 = nn.Sequential(
            nn.Conv2d(1,  25, (1, 10), bias=False),
            nn.Conv2d(25, 25, (n_channels, 1), bias=False),
            nn.BatchNorm2d(25), nn.ELU(),
            nn.MaxPool2d((1, 3), stride=(1, 3)),
            nn.Dropout(dropout),
        )
        self.block2 = conv_block(25, 50,  10)
        self.block3 = conv_block(50, 100, 10)
        self.block4 = conv_block(100, 200, 10, pool=False)

        dummy     = torch.zeros(1, 1, n_channels, n_times)
        feat_size = self._get_feat_size(dummy)
        self.classifier = nn.Linear(feat_size, n_classes)

    def _get_feat_size(self, x):
        try:
            x = self.block1(x); x = self.block2(x)
            x = self.block3(x); x = self.block4(x)
            return x.numel()
        except RuntimeError:
            return 200  # Fallback

    def forward(self, x):
        try:
            x = self.block1(x); x = self.block2(x)
            x = self.block3(x); x = self.block4(x)
        except RuntimeError:
            # Sinyal çok kısaysa son blokları atla
            x = self.block1(x); x = self.block2(x)
        feat = x.view(x.size(0), -1)
        return self.classifier(feat)

    def get_embedding(self, x):
        try:
            x = self.block1(x); x = self.block2(x)
            x = self.block3(x); x = self.block4(x)
        except RuntimeError:
            x = self.block1(x); x = self.block2(x)
        return x.view(x.size(0), -1)


print("EEGNet, ShallowConvNet, DeepConvNet tanımlandı.")

# Test: Tüm modeller düzgün boyut veriyor mu?
x_test = torch.zeros(2, 1, n_channels, n_times)
for ModelClass, name in [(EEGNet, 'EEGNet'), (ShallowConvNet, 'ShallowConvNet'), (DeepConvNet, 'DeepConvNet')]:
    try:
        m = ModelClass(n_channels, n_times)
        out = m(x_test)
        emb = m.get_embedding(x_test)
        params = sum(p.numel() for p in m.parameters())
        print(f"  {name:<18}: çıkış {tuple(out.shape)}, embedding {tuple(emb.shape)}, {params:,} parametre")
    except Exception as e:
        print(f"  {name}: HATA — {e}")

## 5. Eğitim ve Değerlendirme

In [ ]:
def normalize_epochs(X_train, X_test):
    """Kanal bazlı z-normalizasyon (train istatistikleriyle)."""
    mean = X_train.mean(axis=(0, 2), keepdims=True)
    std  = X_train.std(axis=(0, 2), keepdims=True) + 1e-8
    return (X_train - mean) / std, (X_test - mean) / std


def train_model(model, X_train, y_train, X_val, y_val,
                n_epochs, batch_size, lr, patience, device):
    """
    Tek bir fold için model eğitir.
    Early stopping ile aşırı öğrenme önlenir.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    criterion = nn.CrossEntropyLoss()

    # Tensor'lara dönüştür
    Xt = torch.from_numpy(X_train[:, np.newaxis]).float().to(device)   # (n, 1, ch, t)
    yt = torch.from_numpy(y_train).long().to(device)
    Xv = torch.from_numpy(X_val[:, np.newaxis]).float().to(device)
    yv = torch.from_numpy(y_val).long().to(device)

    loader = DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=True)

    best_val_acc = 0.0
    best_state   = None
    no_improve   = 0

    for epoch in range(n_epochs):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(Xv).argmax(dim=1).cpu().numpy()
        val_acc = accuracy_score(yv.cpu().numpy(), val_pred)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve   = 0
        else:
            no_improve += 1

        if no_improve >= patience:
            break

    model.load_state_dict(best_state)
    return model, best_val_acc


def get_embeddings(model, X, device, batch_size=32):
    """Modelin embedding katmanından özellik vektörleri çıkar (PCA için)."""
    model.eval()
    Xt     = torch.from_numpy(X[:, np.newaxis]).float()
    loader = DataLoader(TensorDataset(Xt), batch_size=batch_size)
    embs   = []
    with torch.no_grad():
        for (xb,) in loader:
            embs.append(model.get_embedding(xb.to(device)).cpu().numpy())
    return np.vstack(embs)


print("Eğitim fonksiyonları hazır.")

## 6. Within-Subject 5-Fold CV

Her fold:
1. Train seti üzerinde model eğitilir (early stopping ile)
2. Test seti üzerinde doğruluk ölçülür → **ham model skoru**
3. Train embedding'leri üzerinde PCA fit edilir
4. Test embedding'leri PCA ile dönüştürülür → basit lojistik regresyon → **PCA sonrası skor**

In [ ]:
from sklearn.linear_model import LogisticRegression

MODEL_CLASSES = {
    'EEGNet'       : EEGNet,
    'ShallowConvNet': ShallowConvNet,
    'DeepConvNet'  : DeepConvNet,
}

# {model_name: {'no_pca': [denek_acc, ...], 'pca': [denek_acc, ...]}}
all_results = {name: {'no_pca': [], 'pca': []} for name in MODEL_CLASSES}

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

print(f"Within-subject {N_FOLDS}-fold CV — {len(subjects_X)} denek\n")
print(f"{'Denek':<8}", end='')
for name in MODEL_CLASSES:
    print(f"{name:>18}", end='')
print()
print("-" * (8 + 18 * len(MODEL_CLASSES)))

for sid in subjects_X:
    X_sid = subjects_X[sid]
    y_sid = subjects_y[sid]

    subject_scores = {name: {'no_pca': [], 'pca': []} for name in MODEL_CLASSES}

    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X_sid, y_sid)):
        X_tr_raw, X_te_raw = X_sid[train_idx], X_sid[test_idx]
        y_tr,     y_te     = y_sid[train_idx], y_sid[test_idx]

        # Normalizasyon (train istatistikleri ile)
        X_tr, X_te = normalize_epochs(X_tr_raw, X_te_raw)

        for model_name, ModelClass in MODEL_CLASSES.items():
            # Model oluştur ve eğit
            model = ModelClass(n_channels, n_times).to(DEVICE)
            model, _ = train_model(
                model, X_tr, y_tr, X_te, y_te,
                N_EPOCHS, BATCH_SIZE, LR, PATIENCE, DEVICE
            )

            # ── Boyut azaltma olmadan skor ──
            model.eval()
            with torch.no_grad():
                Xte_t  = torch.from_numpy(X_te[:, np.newaxis]).float().to(DEVICE)
                y_pred = model(Xte_t).argmax(dim=1).cpu().numpy()
            acc_no_pca = accuracy_score(y_te, y_pred)

            # ── PCA + Lojistik Regresyon ──
            emb_train = get_embeddings(model, X_tr, DEVICE)
            emb_test  = get_embeddings(model, X_te, DEVICE)

            # NaN/Inf temizle
            if np.any(np.isnan(emb_train)) or np.any(np.isinf(emb_train)):
                acc_pca = acc_no_pca   # PCA başarısız → ham skoru kullan
            else:
                n_comp  = min(emb_train.shape[1], emb_train.shape[0] - 1, 20)
                pca     = PCA(n_components=n_comp, random_state=42)
                scaler  = StandardScaler()
                emb_tr_pca = pca.fit_transform(scaler.fit_transform(emb_train))
                emb_te_pca = pca.transform(scaler.transform(emb_test))
                lr_clf  = LogisticRegression(max_iter=500, random_state=42)
                try:
                    lr_clf.fit(emb_tr_pca, y_tr)
                    acc_pca = accuracy_score(y_te, lr_clf.predict(emb_te_pca))
                except Exception:
                    acc_pca = acc_no_pca

            subject_scores[model_name]['no_pca'].append(acc_no_pca)
            subject_scores[model_name]['pca'].append(acc_pca)

    # Denek ortalamasını kaydet
    row = f"S{sid:03d}     "
    for name in MODEL_CLASSES:
        mean_acc = np.mean(subject_scores[name]['no_pca'])
        all_results[name]['no_pca'].append(mean_acc)
        mean_pca  = np.mean(subject_scores[name]['pca'])
        all_results[name]['pca'].append(mean_pca)
        row += f"{mean_acc:>18.3f}"
    print(row)

print("\nTamamlandı.")

## 7. Sonuçlar — Boyut Azaltma Olmadan

In [ ]:
print("=" * 60)
print(f"DERİN ÖĞRENME — WITHIN-SUBJECT {N_FOLDS}-FOLD CV")
print("Ham sinyal → uçtan uca öğrenme")
print("=" * 60)

for name in MODEL_CLASSES:
    accs = np.array(all_results[name]['no_pca'])
    print(f"\n{name}:")
    print(f"  Doğruluk : {accs.mean():.4f} ± {accs.std():.4f}")
    print(f"  Medyan   : {np.median(accs):.4f}")
    print(f"  Min/Max  : {accs.min():.3f} / {accs.max():.3f}")

## 8. PCA Sonrası Sonuçlar

In [ ]:
print("=" * 60)
print("PCA + LOJİSTİK REGRESYON — Embedding üzerinde boyut azaltma")
print("=" * 60)

for name in MODEL_CLASSES:
    accs = np.array(all_results[name]['pca'])
    print(f"\n{name} (PCA sonrası):")
    print(f"  Doğruluk : {accs.mean():.4f} ± {accs.std():.4f}")
    delta = accs.mean() - np.mean(all_results[name]['no_pca'])
    print(f"  Değişim  : {delta:+.4f} (PCA öncesine göre)")

## 9. Görselleştirme

In [ ]:
model_names = list(MODEL_CLASSES.keys())
x = np.arange(len(model_names))
w = 0.35

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, key, title in zip(
    axes,
    ['no_pca', 'pca'],
    ['Doğruluk — Boyut Azaltma Olmadan', 'Doğruluk — PCA Sonrası (Embedding)']
):
    vals = [np.mean(all_results[n][key]) for n in model_names]
    stds = [np.std(all_results[n][key]) for n in model_names]
    colors = ['#3498db', '#2ecc71', '#e74c3c']

    bars = ax.bar(x, vals, w*2, yerr=stds, color=colors, alpha=.85,
                  capsize=6, edgecolor='black')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(model_names, fontsize=11)
    ax.set_ylim(0.4, 1.0)
    ax.set_ylabel('Doğruluk')
    ax.axhline(0.5, color='gray', ls='--', alpha=.6, label='Şans (%50)')
    ax.legend(fontsize=9)
    ax.grid(True, axis='y', alpha=.3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.015,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='500')

plt.suptitle(
    f'Derin Öğrenme — Within-Subject {N_FOLDS}-Fold CV ({len(subjects_X)} denek)',
    fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('dl_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Boxplot — denek dağılımı (her model)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, key, title in zip(
    axes,
    ['no_pca', 'pca'],
    ['Denek Dağılımı — PCA Öncesi', 'Denek Dağılımı — PCA Sonrası']
):
    data   = [all_results[n][key] for n in model_names]
    bp     = ax.boxplot(data, labels=model_names, patch_artist=True, notch=False)
    colors = ['#3498db', '#2ecc71', '#e74c3c']
    for patch, c in zip(bp['boxes'], colors):
        patch.set_facecolor(c); patch.set_alpha(.7)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Denek Ortalama Doğruluğu')
    ax.set_ylim(0.3, 1.05)
    ax.axhline(0.5, color='gray', ls='--', alpha=.6, label='Şans')
    ax.legend(fontsize=9); ax.grid(True, axis='y', alpha=.3)

plt.tight_layout()
plt.savefig('dl_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. En İyi Modelin Embedding Analizi

In [ ]:
# En iyi modeli bul
best_model_name = max(MODEL_CLASSES, key=lambda n: np.mean(all_results[n]['no_pca']))
print(f"En iyi model: {best_model_name}")
print(f"Doğruluk    : {np.mean(all_results[best_model_name]['no_pca']):.4f}")

# En iyi denekte embedding görselleştir
best_sid = None
best_acc = 0
for i, sid in enumerate(subjects_X):
    acc = all_results[best_model_name]['no_pca'][i]
    if acc > best_acc:
        best_acc = acc
        best_sid = sid

print(f"\nEn iyi denek: S{best_sid:03d} (acc={best_acc:.3f})")

# Bu denek için model eğit ve embedding çıkar
X_sid = subjects_X[best_sid]
y_sid = subjects_y[best_sid]
X_norm, _ = normalize_epochs(X_sid, X_sid)

model_best = MODEL_CLASSES[best_model_name](n_channels, n_times).to(DEVICE)
model_best, _ = train_model(
    model_best, X_norm, y_sid, X_norm, y_sid,
    N_EPOCHS, BATCH_SIZE, LR, PATIENCE, DEVICE
)

embeddings = get_embeddings(model_best, X_norm, DEVICE)
scaler_emb = StandardScaler()
emb_scaled = scaler_emb.fit_transform(embeddings)

pca_2d = PCA(n_components=2, random_state=42)
emb_2d = pca_2d.fit_transform(emb_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, emb, title in zip(
    axes,
    [emb_2d, emb_scaled[:, :2]],
    [f'{best_model_name} Embedding — 2D PCA\n(S{best_sid:03d})',
     f'{best_model_name} Embedding — Ham İlk 2 Boyut\n(S{best_sid:03d})']
):
    for cls, color, lbl in zip([0, 1], ['#3498db', '#e74c3c'], ['T1 Sol El', 'T2 Sağ El']):
        mask = y_sid == cls
        ax.scatter(emb[mask, 0], emb[mask, 1], c=color, label=lbl, alpha=.6, s=40)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(); ax.grid(alpha=.3)
    ax.set_xlabel('Bileşen 1'); ax.set_ylabel('Bileşen 2')

plt.tight_layout()
plt.savefig('dl_embedding.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPCA açıklanan varyans: {pca_2d.explained_variance_ratio_.sum()*100:.1f}%")

## 11. Karmaşıklık Matrisi — En İyi Model, En İyi Denek

In [ ]:
from sklearn.model_selection import train_test_split

X_sid  = subjects_X[best_sid]
y_sid  = subjects_y[best_sid]

X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(
    X_sid, y_sid, test_size=0.25, random_state=42, stratify=y_sid)
X_tr, X_te = normalize_epochs(X_tr_raw, X_te_raw)

final_model = MODEL_CLASSES[best_model_name](n_channels, n_times).to(DEVICE)
final_model, _ = train_model(
    final_model, X_tr, y_tr, X_te, y_te,
    N_EPOCHS, BATCH_SIZE, LR, PATIENCE, DEVICE
)

final_model.eval()
with torch.no_grad():
    Xte_t  = torch.from_numpy(X_te[:, np.newaxis]).float().to(DEVICE)
    y_pred = final_model(Xte_t).argmax(dim=1).cpu().numpy()

print(f"Sınıflandırma Raporu — {best_model_name} (S{best_sid:03d}):")
print(classification_report(y_te, y_pred, target_names=['T1 Sol El', 'T2 Sağ El']))

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(
    confusion_matrix(y_te, y_pred),
    display_labels=['T1 Sol El', 'T2 Sağ El']
).plot(ax=ax, cmap='Blues')
ax.set_title(f'Karmaşıklık Matrisi — {best_model_name} (S{best_sid:03d})', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('dl_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Klasik ML vs Derin Öğrenme — Nihai Karşılaştırma

Klasik ML sonuçlarını (v4 notebook'tan) bu hücreye girin:

In [ ]:
# ── v4 notebook'tan alınan klasik ML sonuçları ──
# Bu değerleri kendi v4 çıktınızla güncelleyin
classic_ml_results = {
    'SVM (RBF)' : {'no_pca': 0.5622, 'pca': 0.5656},
    'Random Forest': {'no_pca': 0.5505, 'pca': 0.5552},
    'KNN (k=7)' : {'no_pca': 0.5799, 'pca': 0.5710},
}

dl_results_summary = {
    name: {
        'no_pca': np.mean(all_results[name]['no_pca']),
        'pca'   : np.mean(all_results[name]['pca'])
    }
    for name in MODEL_CLASSES
}

# ── Karşılaştırma tablosu ──
print("=" * 75)
print("KLASİK ML vs DERİN ÖĞRENME — WITHIN-SUBJECT KARŞILAŞTIRMA")
print("=" * 75)
print(f"{'Model':<22} {'Tür':<12} {'PCA Öncesi':>12} {'PCA Sonrası':>12} {'Fark':>10}")
print("-" * 75)

for name, res in classic_ml_results.items():
    delta = res['pca'] - res['no_pca']
    print(f"{name:<22} {'Klasik ML':<12} {res['no_pca']:>12.4f} {res['pca']:>12.4f} {delta:>+10.4f}")

print()
for name, res in dl_results_summary.items():
    delta = res['pca'] - res['no_pca']
    print(f"{name:<22} {'Derin Öğr.':<12} {res['no_pca']:>12.4f} {res['pca']:>12.4f} {delta:>+10.4f}")

# ── Görsel karşılaştırma ──
all_models  = list(classic_ml_results.keys()) + list(dl_results_summary.keys())
no_pca_vals = [classic_ml_results[n]['no_pca'] for n in classic_ml_results] +               [dl_results_summary[n]['no_pca'] for n in dl_results_summary]
pca_vals    = [classic_ml_results[n]['pca'] for n in classic_ml_results] +               [dl_results_summary[n]['pca'] for n in dl_results_summary]
colors_bar  = ['#95a5a6'] * 3 + ['#3498db', '#2ecc71', '#e74c3c']

fig, ax = plt.subplots(figsize=(14, 6))
x  = np.arange(len(all_models))
b1 = ax.bar(x - 0.2, no_pca_vals, 0.35, label='PCA Öncesi', color=colors_bar, alpha=.7, edgecolor='black')
b2 = ax.bar(x + 0.2, pca_vals,    0.35, label='PCA Sonrası', color=colors_bar, alpha=1.0, edgecolor='black')

ax.axhline(0.5, color='gray', ls='--', alpha=.7, label='Şans (%50)')
ax.axvline(2.5, color='black', ls=':', alpha=.4)
ax.text(0.9, 0.95, 'Klasik ML', transform=ax.transAxes, ha='right',
        fontsize=10, color='#7f8c8d')
ax.text(0.91, 0.95, '|  Derin Öğrenme', transform=ax.transAxes, ha='left',
        fontsize=10, color='#2c3e50')

ax.set_xticks(x)
ax.set_xticklabels(all_models, rotation=15, ha='right', fontsize=9)
ax.set_ylim(0.4, 1.0)
ax.set_ylabel('Ortalama Doğruluk (Within-Subject)')
ax.set_title('Klasik ML vs Derin Öğrenme — Tüm Modeller', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(True, axis='y', alpha=.3)

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('final_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Tartışma ve Sonuç

### Model Karşılaştırması

| Model | Tür | Özellik | Beklenen Avantaj |
|---|---|---|---|
| SVM / RF / KNN | Klasik ML | CSP özellikleri (12 boyut) | Küçük veri setinde stabil |
| EEGNet | Derin Öğr. | Ham sinyal, depthwise conv | Az parametre, iyi genelleşme |
| ShallowConvNet | Derin Öğr. | Ham sinyal, güç spektrum | CSP'ye yakın, yorumlanabilir |
| DeepConvNet | Derin Öğr. | Ham sinyal, derin hiyerarşi | Büyük veride çok güçlü |

### PCA'nın Derin Öğrenmedeki Rolü

Klasik ML'de PCA özellik uzayını küçültür (640 → 57 boyut).  
Derin öğrenmede ise PCA modelin **öğrendiği temsiller** üzerinde uygulanır.
Bu iki açıdan faydalıdır: embedding'lerin lineer ayrılabilirliğini test eder
ve lojistik regresyon ile basit bir sınıflandırıcının ne kadar başarılı
olduğunu gösterir — yani modelin iç temsillerinin kalitesini ölçer.

### Neden Derin Öğrenme Her Zaman Daha İyi Değil?

Bu veri setinde denek başına yalnızca ~45 epoch vardır.
Derin öğrenme modelleri genellikle yüzlerce-binlerce örnek gerektirir.
45 epoch ile eğitilen bir CNN ile overfitting riski yüksektir;
bu yüzden EEGNet gibi kompakt modeller küçük EEG veri setlerinde
tercih edilir. Daha büyük veri setlerinde (BCI Competition, CHB-MIT) 
DeepConvNet açık ara en iyi sonucu verir.